In [ ]:
import os
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.vector_stores.chroma import ChromaVectorStore
from llama_index.embeddings.openai import OpenAIEmbedding
import chromadb
from dotenv import load_dotenv

In [ ]:
# Load environment variables from .env file
load_dotenv()
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]

In [ ]:
# Reconnect to the existing persistent Chroma database
# This points to the same path where data was saved in the setup phase
chroma_client = chromadb.PersistentClient(path="./chroma_db")

In [ ]:
# Get the existing collection (use get_collection, not get_or_create_collection)
# This retrieves the collection that was created and populated earlier
# Will raise an error if the collection doesn't exist
chroma_collection = chroma_client.get_collection("pdf_collection")

In [ ]:
# Wrap the collection in LlamaIndex's vector store adapter
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

In [ ]:
# Create storage context pointing to the existing vector store
storage_context = StorageContext.from_defaults(vector_store=vector_store)

In [ ]:
# Initialize the same embedding model used when creating the index
# IMPORTANT: Must use the same model, otherwise vectors won't match
embed_model = OpenAIEmbedding(model="text-embedding-3-small")

In [ ]:
# Load the index from the existing vector store
# This reads the previously saved embeddings instead of recomputing them
# Note: from_vector_store() instead of from_documents()
index = VectorStoreIndex.from_vector_store(
    vector_store=vector_store,
    embed_model=embed_model
)

In [ ]:
# Now you can query the index
# The query engine converts your question to a vector and finds similar document chunks
query_engine = index.as_query_engine()
response = query_engine.query("Your question here")
print(response)